# ACCA ARC-AGI-3 Submission v1

Offline-wheel install plus official Swarm execution. No internet calls.

In [ ]:
import glob
import subprocess
import sys
from pathlib import Path

wheel_dirs = sorted({str(Path(p).parent) for p in glob.glob('/kaggle/input/**/arc_agi_3_wheels/*.whl', recursive=True)})
if wheel_dirs:
    find_links = []
    for wheel_dir in wheel_dirs:
        find_links.extend(['--find-links', wheel_dir])
    cmd = [sys.executable, '-m', 'pip', 'install', '--no-index', *find_links, 'arc-agi', '--quiet']
    subprocess.run(cmd, check=True)
else:
    print('No offline ARC-AGI-3 wheels found; assuming SDK is already installed for local smoke tests.')

In [ ]:
import glob
import sys
import zipfile
from pathlib import Path

for zip_path in sorted(glob.glob('/kaggle/input/**/acca_code.zip', recursive=True)):
    target = Path('/kaggle/working/acca_code')
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(target)
    print('Extracted ACCA code from', zip_path)

candidate_roots = [
    Path('/kaggle/working'),
    Path('/kaggle/working/acca_code'),
    Path('/kaggle/working/acca_code/ACCA'),
    Path('/kaggle/input/arc-acca'),
    Path('/kaggle/input/acca'),
    Path.cwd(),
]s
for src_dir in glob.glob('/kaggle/input/**/src', recursive=True):
    candidate_roots.append(Path(src_dir).parent)

found_root = None
for root in candidate_roots:
    if (root / 'src').exists() and str(root) not in sys.path:
        sys.path.insert(0, str(root))
        found_root = root
        break
if found_root is None:
    raise FileNotFoundError('ACCA src/ not found. Attach kaggle/acca_code.zip or a dataset containing src/.')
print('Using ACCA repo root:', found_root)

from src.arc_agi3_bridge import KaggleACCAAgent, run_competition

print('Loaded ACCA Kaggle bridge:', KaggleACCAAgent.__name__)

In [ ]:
import os

if os.environ.get('ACCA_LOCAL_SMOKE') == '1':
    print('Local smoke mode: bridge imported; skipping Kaggle Swarm run.')
else:
    run_competition()